# Retrieval Augmented Generation (RAG) dengan Pendekatan Agent
Notebook ini mendemonstrasikan dua skenario implementasi RAG berbasis Agent (ReAct) menggunakan database vektor Chroma dan model Gemini:
1. **Contoh 1**: RAG menggunakan satu berkas dokumen PDF (`pemilu 2024.pdf`).
2. **Contoh 2**: RAG menggunakan banyak dokumen dengan tipe berkas berbeda (PDF `pemilu 2024.pdf` dan TXT `demografi-jakarta.txt`).

## Contoh 1: RAG dengan Satu Dokumen (PDF)

In [1]:
# 1. Memuat variabel lingkungan
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# 2. Memuat file PDF secara manual menggunakan pypdf
from pypdf import PdfReader
from langchain_core.documents import Document

pdf_file_path = "data/pemilu 2024.pdf"
reader = PdfReader(pdf_file_path)

docs_single = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    docs_single.append(Document(page_content=text, metadata={"source": pdf_file_path, "page": i}))

print(f"Total halaman dokumen: {len(docs_single)}")

Total halaman dokumen: 2


In [3]:
# 3. Membagi dokumen menjadi beberapa chunk/potongan teks kecil
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
splits_single = text_splitter.split_documents(docs_single)

print(f"Total Chunks : {len(splits_single)}")

Total Chunks : 5


In [4]:
# 4. Inisialisasi Model LLM dan Embeddings dari Google GenAI
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    max_retries=2,
)

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
)

In [5]:
# 5. Membuat database vektor Chroma (in-memory) dan retriever
from langchain_chroma import Chroma

vectorstore_single = Chroma.from_documents(
    documents=splits_single,
    embedding=embeddings,
)

retriever_single = vectorstore_single.as_retriever(
    search_kwargs={"k": 4}
)

print("Vector Database Single Ready")

Vector Database Single Ready


In [6]:
# 6. Mendefinisikan tool pencarian dokumen tunggal
from langchain_core.tools import tool

@tool
def cari_informasi_pemilu(query: str) -> str:
    """Cari dan dapatkan informasi relevan tentang Pemilu 2024 dari dokumen resmi."""
    docs = retriever_single.invoke(query)
    return "\n\n".join(doc.page_content for doc in docs)

In [7]:
# 7. Membuat Agent untuk RAG Dokumen Tunggal
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

system_prompt_single = """
Kamu adalah asisten BPS.
Jawab pertanyaan hanya berdasarkan informasi yang diperoleh dari tool yang tersedia.
Jika jawaban tidak ditemukan, jawab: 'Maaf, saya tidak menemukan informasi tersebut pada dokumen.'
Jangan menggunakan pengetahuan di luar dokumen resmi.
Jawab maksimal 3 kalimat.
"""

agent_single = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[cari_informasi_pemilu],
    system_prompt=system_prompt_single
)

In [8]:
# 8. Menguji Agent Dokumen Tunggal
question = HumanMessage(content="kapan pemilu 2024?")
response = agent_single.invoke({"messages": [question]})

print("Pertanyaan:")
print(question.content)
print("\nJawaban Agent:")
print(response["messages"][-1].content)

Pertanyaan:
kapan pemilu 2024?

Jawaban Agent:
[{'type': 'text', 'text': 'Pemungutan suara Pemilihan Umum 2024 telah dilaksanakan pada Rabu, 14 Februari 2024. Sementara itu, pemilihan gubernur, bupati, dan wali kota disepakati digelar pada 27 November 2024.', 'extras': {'signature': 'EjQKMgERTTIPS3xaTZne9xmxyvFcvfsC7HQXTDkStujJwZCEJBwMdTQtmHIJeQb9WT/lTC2n'}}]


## Contoh 2: RAG dengan Banyak Dokumen (Multifile - PDF & TXT)

In [9]:
# 1. Memuat beberapa file berbeda (PDF dan TXT)
import os
from pypdf import PdfReader
from langchain_core.documents import Document

docs_multi = []

# A. Memuat PDF
pdf_path = "data/pemilu 2024.pdf"
if os.path.exists(pdf_path):
    print(f"Memuat dokumen PDF: {pdf_path}")
    reader = PdfReader(pdf_path)
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        docs_multi.append(Document(page_content=text, metadata={"source": pdf_path, "page": i}))

# B. Memuat TXT
txt_path = "data/demografi-jakarta.txt"
if os.path.exists(txt_path):
    print(f"Memuat dokumen TXT: {txt_path}")
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read()
    docs_multi.append(Document(page_content=text, metadata={"source": txt_path}))

print(f"Total halaman/dokumen yang berhasil dimuat: {len(docs_multi)}")

Memuat dokumen PDF: data/pemilu 2024.pdf
Memuat dokumen TXT: data/demografi-jakarta.txt
Total halaman/dokumen yang berhasil dimuat: 3


In [10]:
# 2. Membagi dokumen-dokumen menjadi beberapa chunk/potongan teks kecil
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
splits_multi = text_splitter.split_documents(docs_multi)

print(f"Total Chunks : {len(splits_multi)}")

Total Chunks : 8


In [11]:
# 3. Membuat database vektor Chroma (dengan penyimpanan lokal/persist) dan retriever
import shutil
from langchain_chroma import Chroma

persist_directory = "vector_multi"

# Bersihkan folder vector lama jika ingin membuat ulang database vektor dari awal
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)

vectorstore_multi = Chroma.from_documents(
    documents=splits_multi,
    embedding=embeddings,
    persist_directory=persist_directory
)

retriever_multi = vectorstore_multi.as_retriever(
    search_kwargs={"k": 4}
)

print("Vector Database Multiple Files Ready (Persisted locally)")

Vector Database Multiple Files Ready (Persisted locally)


In [12]:
# 4. Mendefinisikan tool pencarian multi-dokumen
from langchain_core.tools import tool

@tool
def cari_informasi_multi_dokumen(query: str) -> str:
    """Cari dan dapatkan informasi relevan tentang Pemilu 2024 atau Demografi Jakarta dari dokumen resmi."""
    docs = retriever_multi.invoke(query)
    return "\n\n".join(doc.page_content for doc in docs)

In [13]:
# 5. Membuat Agent untuk RAG Multi-Dokumen
from langchain.agents import create_agent

system_prompt_multi = """
Kamu adalah asisten BPS.
Jawab pertanyaan hanya berdasarkan informasi yang diperoleh dari tool yang tersedia.
Jika jawaban tidak ditemukan, jawab: 'Maaf, saya tidak menemukan informasi tersebut pada dokumen.'
Jangan menggunakan pengetahuan di luar dokumen resmi.
Jawab maksimal 3 kalimat.
"""

agent_multi = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[cari_informasi_multi_dokumen],
    system_prompt=system_prompt_multi
)

In [14]:
# 6. Menguji Agent Multi-Dokumen dengan beberapa kueri topik berbeda
from langchain_core.messages import HumanMessage

queries = [
    "kapan pemilu 2024?",
    "berapa persentase suku jawa di jakarta?"
]

for query in queries:
    print(f"=== Pertanyaan: {query} ===")
    question = HumanMessage(content=query)
    response = agent_multi.invoke({"messages": [question]})
    print("Jawaban Agent:")
    print(response["messages"][-1].content)
    print("-" * 50)

=== Pertanyaan: kapan pemilu 2024? ===


Jawaban Agent:
[{'type': 'text', 'text': 'Pemungutan suara Pemilihan Umum (Pemilu) 2024 dilaksanakan pada Rabu, 14 Februari 2024. Sementara itu, pemilihan gubernur, bupati, dan walikota disepakati digelar pada 27 November 2024.', 'extras': {'signature': 'EjQKMgERTTIPPOxifVY4vyJepNRDaixAtL82Hc0EtkQWVu5KxUz9Vs67fHLojHlkWixYM/kj'}}]
--------------------------------------------------
=== Pertanyaan: berapa persentase suku jawa di jakarta? ===


Jawaban Agent:
[{'type': 'text', 'text': 'Berdasarkan data demografi penduduk Jakarta tahun 2024, persentase suku Jawa di Jakarta adalah sebesar 35,16%.', 'extras': {'signature': 'EjQKMgERTTIPzA7Yte/0hAVMQrWC2jo6f+LfPQBIPQknUHrs+3UeKqCrcWwrSQs/RhIfhnqg'}}]
--------------------------------------------------
